# **Notebook 11 – Handling Imbalanced Data**

In [23]:
import pandas as pd 
df = pd.read_csv("credit_card_fraud_imbalanced_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Transaction_ID             5000 non-null   object 
 1   Customer_Age               5000 non-null   int64  
 2   Annual_Income              5000 non-null   float64
 3   Transaction_Amount         5000 non-null   float64
 4   Transactions_Last_30_Days  5000 non-null   int64  
 5   Distance_From_Home_Km      5000 non-null   float64
 6   Online_Transaction         5000 non-null   object 
 7   Device_Type                5000 non-null   object 
 8   Merchant_Category          5000 non-null   object 
 9   International_Transaction  5000 non-null   object 
 10  Previous_Fraud_Count       5000 non-null   int64  
 11  Account_Age_Months         5000 non-null   int64  
 12  Fraud                      5000 non-null   int64  
dtypes: float64(3), int64(5), object(5)
memory usage:

## **1. What is Class Imbalance?**



**Understand the Concept**

- Class imbalance occurs when one class has many more records than another class.
- The class with more records is called the majority class.
- The class with fewer records is called the minority class.
- Imbalanced datasets are common in classification problems.

**Demonstrate the Concept**

**Example:**
- Check the distribution of the Fraud target variable.

**AI/ML Usage:**
- Helps identify whether the classification dataset is imbalanced.

In [24]:
# Implement the Concept
# class distribution
print(df["Fraud"].value_counts())

Fraud
0    4798
1     202
Name: count, dtype: int64


In [25]:
# class percentages
print(df["Fraud"].value_counts(normalize=True) * 100)

Fraud
0    95.96
1     4.04
Name: proportion, dtype: float64


**Explanation**

- value_counts() shows the number of records in each class.
- The majority class has more records than the minority class.
- The percentages show how imbalanced the Fraud target is.

## **2. Balanced vs Imbalanced Dataset**



**Understand the Concept**

- A balanced dataset has a similar number of records in each class.
- An imbalanced dataset has a large difference in the number of records between classes.
- Balanced data gives classes a more equal representation during model training.

**Demonstrate the Concept**

**Example:**
- Compare the Merchant Category and fraud col.


In [26]:
# Implement the Concept

# Compare col
print("Imbalance col \n",df["Fraud"].value_counts())
print("\nBalance col \n",df["Merchant_Category"].value_counts())

Imbalance col 
 Fraud
0    4798
1     202
Name: count, dtype: int64

Balance col 
 Merchant_Category
Dining         760
Grocery        757
Clothing       736
Electronics    723
Healthcare     695
Fuel           668
Travel         661
Name: count, dtype: int64


**Explanation**

- value_counts() shows the number of records in each class.
- A small difference indicates a more balanced dataset.
- A large difference indicates an imbalanced dataset.

## **3. Why Class Imbalance is a Problem**



**Understand the Concept**

- The majority class can dominate the model during training.
- The model may perform well on the majority class but poorly on the minority class.
- Accuracy alone can be misleading for imbalanced datasets.
- Minority-class cases are often the most important cases to detect.

**Demonstrate the Concept**

**Example:**
- Calculate accuracy when a model predicts every transaction as Normal.

**AI/ML Usage:**
- Helps understand why minority-class performance should also be evaluated.

In [27]:
# Implement the Concept
print("Accuracy:", (df["Fraud"] == 0).sum()/ len(df))

Accuracy: 0.9596


**Explanation**

- The model predicts every transaction as Normal.
- Because Normal transactions are the majority class, accuracy can still be high.
- However, the model detects no Fraud transactions.
- Therefore, accuracy alone can give a misleading result.

## **4. Undersampling**



**Understand the Concept**

- Undersampling reduces the number of records in the majority class.
- It creates a more balanced class distribution.
- Some majority-class information is removed during this process.

**Demonstrate the Concept**

**Example:**
- Reduce Normal transactions to match the number of Fraud transactions.

**AI/ML Usage:**
- Helps create a balanced training dataset when the majority class is very large.

In [28]:
# Implement the Concept
from sklearn.utils import resample

normal = df[df["Fraud"] == 0]
fraud = df[df["Fraud"] == 1]

# Undersample the majority class
normal_downsampled = resample(normal, replace=False, n_samples=len(fraud), random_state=42)
df_undersampled = pd.concat([normal_downsampled, fraud])
print(df_undersampled["Fraud"].value_counts())

Fraud
0    202
1    202
Name: count, dtype: int64


**Explanation**

- Normal transactions are reduced to the same size as Fraud transactions.
- No new records are created.
- Some majority-class information is lost.

## **5. Oversampling**



**Understand the Concept**

- Oversampling increases the number of records in the minority class.
- It gives the minority class more representation during training.
- Oversampling keeps the majority-class records unchanged.

**Demonstrate the Concept**

**Example:**
- Increase Fraud transactions to match the number of Normal transactions.

**AI/ML Usage:**
- Helps models learn more effectively from minority-class records.

In [29]:
# Implement the Concept
# Oversample the minority class
fraud_upsampled = resample(fraud,replace=True, n_samples=len(normal), random_state=42)
df_oversampled = pd.concat([normal, fraud_upsampled])
print(df_oversampled["Fraud"].value_counts())

Fraud
0    4798
1    4798
Name: count, dtype: int64


**Explanation**

- Fraud transactions are increased to match Normal transactions.
- Existing minority records are reused.
- This creates a more balanced training dataset.

## **6. Random Oversampling**



**Understand the Concept**

- Random Oversampling randomly duplicates minority-class records.
- It increases the size of the minority class without removing majority records.
- It is simple but can increase the risk of overfitting.

**Demonstrate the Concept**

**Example:**
- Randomly duplicate Fraud transactions until both classes have equal records.

**AI/ML Usage:**
- Provides a simple way to increase minority-class representation.

In [30]:
# Implement the Concept
from imblearn.over_sampling import RandomOverSampler

X = df.drop(columns=["Fraud"])
y = df["Fraud"]
X = X.select_dtypes(include="number")

# Random Oversampling
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X, y)
print(y_resampled.value_counts())

Fraud
0    4798
1    4798
Name: count, dtype: int64


**Explanation**

- RandomOverSampler duplicates minority-class records randomly.
- The minority class becomes equal to the majority class.
- No synthetic records are generated.

## **7. Random Undersampling**



**Understand the Concept**

- Random Undersampling randomly removes records from the majority class.
- It reduces the difference between the classes.
- It is simple and can make model training faster.

**Demonstrate the Concept**

**Example:**
- Randomly remove Normal transactions until both classes have equal records.

**AI/ML Usage:**
- Useful when the majority class is much larger than the minority class.

In [31]:
# Implement the Concept
from imblearn.under_sampling import RandomUnderSampler
# Random Undersampling
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)
print(y_resampled.value_counts())

Fraud
0    202
1    202
Name: count, dtype: int64


**Explanation**

- RandomUnderSampler removes majority-class records randomly.
- Both classes become equally represented.
- Some Normal transaction information is discarded.

## **8. SMOTE**



**Understand the Concept**

- SMOTE stands for Synthetic Minority Over-sampling Technique.
- It creates new synthetic minority-class records.
- It uses existing minority-class records to generate new samples.
- Unlike random oversampling, it does not simply duplicate records.

**Demonstrate the Concept**

**Example:**
- Generate synthetic Fraud transactions using SMOTE.

**AI/ML Usage:**
- Helps models learn minority-class patterns from additional synthetic samples.

In [32]:
# Implement the Concept
from imblearn.over_sampling import SMOTE

# SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
print(y_resampled.value_counts())

Fraud
0    4798
1    4798
Name: count, dtype: int64


**Explanation**

- SMOTE creates synthetic samples for the Fraud class.
- The minority class is increased without simple duplication.
- This helps provide more balanced training data.

## **9. Borderline-SMOTE**



**Understand the Concept**

- Borderline-SMOTE is a variation of SMOTE.
- It focuses on minority-class samples near the class boundary.
- These samples are often harder for the model to classify correctly.

**Demonstrate the Concept**

**Example:**
- Generate synthetic Fraud samples near difficult class boundaries.

**AI/ML Usage:**
- Helps improve learning around challenging minority-class regions.

In [33]:
# Implement the Concept
from imblearn.over_sampling import BorderlineSMOTE

# Borderline-SMOTE
borderline_smote = BorderlineSMOTE(random_state=42)
X_resampled, y_resampled = borderline_smote.fit_resample(X, y)
print(y_resampled.value_counts())

Fraud
0    4798
1    4798
Name: count, dtype: int64


**Explanation**

- Borderline-SMOTE identifies minority samples near difficult class boundaries.
- Synthetic samples are generated around these areas.
- This focuses learning on harder classification cases.

## **10. Class Weights**



**Understand the Concept**

- Class weights give different importance to different classes.
- The minority class can receive a higher weight.
- The model can learn from imbalanced data without changing the original records.

**Demonstrate the Concept**

**Example:**
- Train a Logistic Regression model using balanced class weights.

**AI/ML Usage:**
- Helps the model give more importance to minority-class errors.

In [34]:
# Implement the Concept
from sklearn.linear_model import LogisticRegression

# Train model with balanced class weights
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X, y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


**Explanation**

- class_weight="balanced" automatically gives more weight to the minority class.
- The original class distribution is not changed.
- The model gives greater importance to minority-class errors.

In [35]:
# Evaluate the model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
y_pred = model.predict(X)

accuracy = accuracy_score(y, y_pred)
print("Accuracy:", round(accuracy, 4))
print("\nClassification Report:\n",classification_report(y, y_pred))
print("\nConfusion Matrix:\n",confusion_matrix(y, y_pred))

Accuracy: 0.5094

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.51      0.66      4798
           1       0.05      0.57      0.09       202

    accuracy                           0.51      5000
   macro avg       0.51      0.54      0.38      5000
weighted avg       0.93      0.51      0.64      5000


Confusion Matrix:
 [[2432 2366]
 [  87  115]]


**Explanation**

- **Accuracy** shows the overall percentage of correct predictions.
- **Precision** shows how many predicted Fraud transactions were actually Fraud.
- **Recall** shows how many actual Fraud transactions were detected.
- **F1-Score** balances Precision and Recall.
- **Confusion Matrix** shows correct and incorrect predictions for each class.
- For imbalanced data, **Recall and F1-Score for the minority class are especially important**, not just accuracy.

## **11. When to Use Each Technique**



**Understand the Concept**

- **Undersampling:** Use when the majority class has many more records and removing some of them will not cause significant information loss.
- **Oversampling:** Use when the minority class has too few records and you want to increase its representation without removing majority records.
- **Random Oversampling:** Use when you need a simple and quick method to balance the classes by duplicating minority records.
- **Random Undersampling:** Use when the majority class is very large and you can afford to remove some majority records.
- **SMOTE:** Use when you need new synthetic minority samples instead of simply duplicating existing records.
- **Borderline-SMOTE:** Use when minority-class samples near the decision boundary are difficult to classify.
- **Class Weights:** Use when you want the model to give more importance to the minority class without changing the dataset.
- **Accuracy:** Do not rely on accuracy alone when classes are highly imbalanced.
- **Precision, Recall, F1-Score, and ROC-AUC:** Use additional evaluation metrics to understand minority-class performance.
- **Training Data:** Apply resampling techniques only to the training data, not to the test data.
- **Data Leakage:** Avoid generating synthetic or duplicated samples before splitting the dataset.

